# Chapter 13 Lab — Observability

**Book:** Agentic AI: Build, Ship, Portfolio  
**Chapter:** 13 — Observability  
**Goal:** Instrument agents with traces, structured logs, and metrics so you can understand, debug, and improve them in production.

### What you will do in this lab

| Section | Topic |
|---------|-------|
| 1 | Why Agents Need Observability |
| 2 | Traces and Spans with OpenTelemetry |
| 3 | Structured Logging |
| 4 | Metrics That Matter |
| 5 | Instrumenting an Agent |
| 6 | The Observability Stack |
| 7 | Debugging Agent Behavior |
| 8 | Building Dashboards |
| 9 | LangSmith and Langfuse Integration Patterns |
| 10 | Exercises |

> **Prerequisites:** Python 3.10+. All code runs locally — no external services required.

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
# Run once to install required packages (restart the kernel afterwards).

%pip install opentelemetry-api opentelemetry-sdk langsmith --quiet

In [ ]:
import json
import logging
import random
import statistics
import sys
import textwrap
import time
import uuid
from collections import defaultdict
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from io import StringIO
from typing import Any, Optional

print("Imports OK")

---
## 1. Why Agents Need Observability

Traditional software is deterministic: the same input produces the same output, and a stack trace tells you what went wrong. Agents are different:

- **Non-deterministic** — the same prompt can yield different tool-call sequences.
- **Multi-step** — a failure on step 5 may be caused by a bad decision on step 2.
- **Opaque** — LLM reasoning is hidden inside completion responses.

Without observability you have a **black box**. Let's see what that looks like.

In [ ]:
# ── The Black Box Agent ──────────────────────────────────────────────────────
# A simulated agent with zero observability. When it fails, good luck debugging.

def black_box_agent(query: str) -> str:
    """Simulated agent: retrieves docs, reasons, produces answer."""
    # Step 1: Retrieve (simulated)
    docs = ["Revenue grew 12% YoY.", "Operating margin was 18%."]
    
    # Step 2: Reason (simulated LLM call — sometimes fails)
    time.sleep(random.uniform(0.1, 0.5))  # latency
    if random.random() < 0.3:
        raise RuntimeError("LLM returned empty response")  # intermittent failure
    
    # Step 3: Answer
    return f"Based on the data: {docs[0]} {docs[1]}"


# Run it a few times — sometimes it works, sometimes it crashes.
for i in range(5):
    try:
        result = black_box_agent("Summarize Q3 earnings")
        print(f"Run {i+1}: OK — {result[:60]}...")
    except Exception as e:
        print(f"Run {i+1}: FAILED — {e}")
        # With no traces or logs, we have NO idea what happened before the crash.

print("\n--- With zero observability, intermittent failures are invisible. ---")

In [ ]:
# ── The Instrumented Version ─────────────────────────────────────────────────
# Same logic, but now every step emits structured data.

def instrumented_agent(query: str) -> dict:
    """Same agent, now with observability baked in."""
    trace = {
        "trace_id": str(uuid.uuid4())[:8],
        "query": query,
        "steps": [],
        "status": "success",
        "start_time": time.time(),
    }
    
    # Step 1: Retrieve
    t0 = time.time()
    docs = ["Revenue grew 12% YoY.", "Operating margin was 18%."]
    trace["steps"].append({
        "step": "retrieve",
        "duration_ms": round((time.time() - t0) * 1000, 2),
        "doc_count": len(docs),
    })
    
    # Step 2: Reason
    t0 = time.time()
    latency = random.uniform(0.1, 0.5)
    time.sleep(latency)
    if random.random() < 0.3:
        trace["steps"].append({
            "step": "reason",
            "duration_ms": round((time.time() - t0) * 1000, 2),
            "error": "LLM returned empty response",
        })
        trace["status"] = "error"
        trace["total_ms"] = round((time.time() - trace["start_time"]) * 1000, 2)
        return trace
    
    trace["steps"].append({
        "step": "reason",
        "duration_ms": round((time.time() - t0) * 1000, 2),
        "tokens_used": random.randint(150, 400),
    })
    
    # Step 3: Answer
    trace["answer"] = f"Based on the data: {docs[0]} {docs[1]}"
    trace["total_ms"] = round((time.time() - trace["start_time"]) * 1000, 2)
    return trace


# Run the instrumented version
for i in range(5):
    result = instrumented_agent("Summarize Q3 earnings")
    status_icon = "OK" if result["status"] == "success" else "FAIL"
    print(f"Run {i+1} [{status_icon}] trace={result['trace_id']} "
          f"total={result['total_ms']}ms steps={len(result['steps'])}")

# Show full trace of last run
print("\n--- Full trace of last run: ---")
print(json.dumps(result, indent=2, default=str))

**Key insight:** The instrumented agent captures the same information a debugger would — but automatically, for every run, in production. That is observability.

---
## 2. Traces and Spans with OpenTelemetry

OpenTelemetry (OTel) is the industry standard for distributed tracing. The core concepts:

| Concept | What it means |
|---------|---------------|
| **Trace** | End-to-end journey of a single request |
| **Span** | One unit of work within a trace (e.g., "retrieve documents") |
| **Attributes** | Key-value metadata attached to a span |
| **Events** | Timestamped log entries within a span |

Let's set up an in-memory OTel tracer so we can inspect spans without running a collector.

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory import InMemorySpanExporter

# ── Configure in-memory exporter (no external collector needed) ────────────
exporter = InMemorySpanExporter()
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(exporter))
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("agent-observability-lab")

print("OpenTelemetry tracer configured with in-memory exporter.")
print("All spans will be captured locally for inspection.")

In [ ]:
# ── Basic tracing: wrap agent steps in spans ─────────────────────────────────

def traced_agent_run(query: str) -> str:
    """An agent run where each step is an OpenTelemetry span."""
    with tracer.start_as_current_span("agent_run") as root_span:
        root_span.set_attribute("agent.query", query)
        root_span.set_attribute("agent.model", "gpt-4o-mini")
        
        # Step 1: Retrieve
        with tracer.start_as_current_span("retrieve") as span:
            time.sleep(0.05)  # simulate retrieval latency
            docs = ["Revenue grew 12% YoY.", "Operating margin was 18%."]
            span.set_attribute("retrieve.doc_count", len(docs))
            span.set_attribute("retrieve.source", "vector_store")
            span.add_event("documents_retrieved", {"count": len(docs)})
        
        # Step 2: Reason
        with tracer.start_as_current_span("llm_call") as span:
            time.sleep(random.uniform(0.1, 0.3))  # simulate LLM latency
            tokens = random.randint(200, 500)
            span.set_attribute("llm.model", "gpt-4o-mini")
            span.set_attribute("llm.tokens.total", tokens)
            span.set_attribute("llm.tokens.prompt", int(tokens * 0.7))
            span.set_attribute("llm.tokens.completion", int(tokens * 0.3))
            answer = f"Summary: {docs[0]} {docs[1]}"
            span.add_event("llm_response_received")
        
        # Step 3: Format output
        with tracer.start_as_current_span("format_output") as span:
            final = f"[Agent] {answer}"
            span.set_attribute("output.length", len(final))
        
        root_span.set_attribute("agent.status", "success")
        return final


result = traced_agent_run("Summarize Q3 earnings")
print(result)

In [ ]:
# ── Inspect the captured spans ───────────────────────────────────────────────

def display_spans(exporter: InMemorySpanExporter):
    """Pretty-print all captured spans."""
    spans = exporter.get_finished_spans()
    print(f"Captured {len(spans)} spans:\n")
    
    for s in spans:
        duration_ms = (s.end_time - s.start_time) / 1_000_000  # ns to ms
        parent = s.parent.span_id if s.parent else None
        print(f"  Span: {s.name}")
        print(f"    ID:       {format(s.context.span_id, '016x')}")
        print(f"    Parent:   {format(parent, '016x') if parent else 'ROOT'}")
        print(f"    Duration: {duration_ms:.1f} ms")
        if s.attributes:
            for k, v in s.attributes.items():
                print(f"    {k}: {v}")
        if s.events:
            for ev in s.events:
                print(f"    Event: {ev.name} {dict(ev.attributes) if ev.attributes else ''}")
        print()


display_spans(exporter)

In [ ]:
# ── Visualize the span tree ──────────────────────────────────────────────────

def print_span_tree(exporter: InMemorySpanExporter):
    """Print spans as an indented tree to show parent-child relationships."""
    spans = exporter.get_finished_spans()
    
    # Build lookup
    by_id = {s.context.span_id: s for s in spans}
    children = defaultdict(list)
    roots = []
    for s in spans:
        if s.parent:
            children[s.parent.span_id].append(s)
        else:
            roots.append(s)
    
    def _print(span, indent=0):
        dur = (span.end_time - span.start_time) / 1_000_000
        prefix = "  " * indent
        print(f"{prefix}[{dur:6.1f} ms] {span.name}")
        for child in children.get(span.context.span_id, []):
            _print(child, indent + 1)
    
    print("Span Tree:")
    print("=" * 50)
    for root in roots:
        _print(root)


print_span_tree(exporter)
exporter.clear()  # reset for next section

---
## 3. Structured Logging

Plain-text logs (`print` statements) are almost useless at scale. Structured logs emit **JSON objects** that can be parsed, queried, and aggregated by tools like Elasticsearch, Loki, or CloudWatch.

Key principles:
- Every log line is a JSON object.
- Always include `timestamp`, `level`, `trace_id`, and `step`.
- Never log secrets or PII.

In [ ]:
# ── Structured Logger ────────────────────────────────────────────────────────

class AgentLogger:
    """JSON structured logger for agent runs."""
    
    def __init__(self, agent_name: str, stream=None):
        self.agent_name = agent_name
        self.stream = stream or sys.stdout
        self._buffer: list[dict] = []  # keep a copy for analysis
    
    def _emit(self, level: str, message: str, trace_id: str = "", **extra):
        entry = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "level": level,
            "agent": self.agent_name,
            "trace_id": trace_id,
            "message": message,
            **extra,
        }
        self._buffer.append(entry)
        self.stream.write(json.dumps(entry) + "\n")
    
    def info(self, message: str, **kwargs):
        self._emit("INFO", message, **kwargs)
    
    def warning(self, message: str, **kwargs):
        self._emit("WARNING", message, **kwargs)
    
    def error(self, message: str, **kwargs):
        self._emit("ERROR", message, **kwargs)
    
    def debug(self, message: str, **kwargs):
        self._emit("DEBUG", message, **kwargs)
    
    def get_logs(self, level: Optional[str] = None) -> list[dict]:
        """Retrieve buffered logs, optionally filtered by level."""
        if level:
            return [e for e in self._buffer if e["level"] == level]
        return list(self._buffer)


# ── Demo ──────────────────────────────────────────────────────────────────────
logger = AgentLogger("earnings-agent")
tid = str(uuid.uuid4())[:8]

logger.info("Agent run started", trace_id=tid, query="Summarize Q3 earnings")
logger.info("Documents retrieved", trace_id=tid, step="retrieve", doc_count=3)
logger.info("LLM call completed", trace_id=tid, step="reason", tokens=312, latency_ms=245)
logger.warning("High token usage detected", trace_id=tid, step="reason", tokens=312, threshold=300)
logger.info("Agent run completed", trace_id=tid, status="success", total_ms=310)

In [ ]:
# ── Querying structured logs ─────────────────────────────────────────────────
# Because logs are dicts, we can filter/aggregate them programmatically.

print("=== All WARNING+ logs ===")
for entry in logger.get_logs(level="WARNING"):
    print(json.dumps(entry, indent=2))

print("\n=== Extract latency from logs ===")
for entry in logger.get_logs():
    if "latency_ms" in entry:
        print(f"  trace={entry['trace_id']} step={entry.get('step')} latency={entry['latency_ms']}ms")

---
## 4. Metrics That Matter

Not all numbers are useful. For agents, focus on these core metrics:

| Metric | Why it matters |
|--------|----------------|
| **Latency (p50, p95, p99)** | User experience; SLA compliance |
| **Token usage** | Cost control |
| **Success rate** | Reliability |
| **Steps per run** | Efficiency — more steps = higher cost and latency |
| **Tool error rate** | Identifies flaky integrations |
| **Retrieval relevance** | RAG quality signal |

In [ ]:
# ── Metrics Collector ────────────────────────────────────────────────────────

class MetricsCollector:
    """Lightweight in-memory metrics collector for agent runs."""
    
    def __init__(self):
        self.counters: dict[str, int] = defaultdict(int)
        self.histograms: dict[str, list[float]] = defaultdict(list)
        self.gauges: dict[str, float] = {}
    
    def increment(self, name: str, value: int = 1):
        """Increment a counter (e.g., total_runs, errors)."""
        self.counters[name] += value
    
    def record(self, name: str, value: float):
        """Record a value in a histogram (e.g., latency, tokens)."""
        self.histograms[name].append(value)
    
    def set_gauge(self, name: str, value: float):
        """Set a gauge to a current value (e.g., active_runs)."""
        self.gauges[name] = value
    
    def get_percentile(self, name: str, p: float) -> Optional[float]:
        """Get a percentile from a histogram."""
        values = sorted(self.histograms.get(name, []))
        if not values:
            return None
        idx = int(len(values) * p / 100)
        return values[min(idx, len(values) - 1)]
    
    def summary(self) -> dict:
        """Generate a summary report of all metrics."""
        report = {"counters": dict(self.counters), "gauges": dict(self.gauges)}
        report["histograms"] = {}
        for name, values in self.histograms.items():
            if values:
                report["histograms"][name] = {
                    "count": len(values),
                    "mean": round(statistics.mean(values), 2),
                    "p50": round(self.get_percentile(name, 50), 2),
                    "p95": round(self.get_percentile(name, 95), 2),
                    "p99": round(self.get_percentile(name, 99), 2),
                    "min": round(min(values), 2),
                    "max": round(max(values), 2),
                }
        return report


metrics = MetricsCollector()
print("MetricsCollector ready.")

In [ ]:
# ── Simulate 50 agent runs and collect metrics ───────────────────────────────

random.seed(42)

for i in range(50):
    # Simulate varying conditions
    latency = random.gauss(300, 80)  # ms, mean=300, std=80
    tokens = random.randint(150, 600)
    steps = random.choice([2, 3, 3, 3, 4, 4, 5])
    success = random.random() > 0.1  # 90% success rate
    
    metrics.increment("agent.runs.total")
    metrics.record("agent.latency_ms", max(latency, 50))  # clamp
    metrics.record("agent.tokens", tokens)
    metrics.record("agent.steps", steps)
    
    if success:
        metrics.increment("agent.runs.success")
    else:
        metrics.increment("agent.runs.error")

# Show the summary
summary = metrics.summary()
print(json.dumps(summary, indent=2))

# Compute derived metrics
total = summary["counters"]["agent.runs.total"]
success = summary["counters"].get("agent.runs.success", 0)
print(f"\nSuccess rate: {success}/{total} = {success/total*100:.1f}%")
print(f"p50 latency:  {summary['histograms']['agent.latency_ms']['p50']} ms")
print(f"p95 latency:  {summary['histograms']['agent.latency_ms']['p95']} ms")

---
## 5. Instrumenting an Agent

Now we combine all three pillars — traces, logs, metrics — into a single decorator that wraps any agent function. This is the pattern you will use in production.

In [ ]:
# ── Observable Agent Wrapper ─────────────────────────────────────────────────

class ObservableAgent:
    """Wraps an agent with full observability: traces, logs, metrics."""
    
    def __init__(self, name: str):
        self.name = name
        self.logger = AgentLogger(name)
        self.metrics = MetricsCollector()
        self.tracer = trace.get_tracer(f"{name}-tracer")
        self.runs: list[dict] = []  # store full run records
    
    def run(self, query: str) -> dict:
        """Execute an instrumented agent run."""
        trace_id = str(uuid.uuid4())[:8]
        self.metrics.increment("runs.total")
        self.logger.info("Run started", trace_id=trace_id, query=query)
        
        run_record = {
            "trace_id": trace_id,
            "query": query,
            "steps": [],
            "status": "success",
        }
        
        with self.tracer.start_as_current_span("agent_run") as root_span:
            root_span.set_attribute("trace_id", trace_id)
            root_span.set_attribute("query", query)
            start = time.time()
            
            try:
                # Step 1: Plan
                step = self._execute_step(trace_id, "plan", self._plan, query)
                run_record["steps"].append(step)
                
                # Step 2: Retrieve
                step = self._execute_step(trace_id, "retrieve", self._retrieve, query)
                run_record["steps"].append(step)
                docs = step["result"]
                
                # Step 3: Reason
                step = self._execute_step(trace_id, "reason", self._reason, query, docs)
                run_record["steps"].append(step)
                
                run_record["answer"] = step["result"]
                self.metrics.increment("runs.success")
                
            except Exception as e:
                run_record["status"] = "error"
                run_record["error"] = str(e)
                self.metrics.increment("runs.error")
                self.logger.error(f"Run failed: {e}", trace_id=trace_id)
                root_span.set_attribute("error", True)
                root_span.set_attribute("error.message", str(e))
            
            elapsed_ms = (time.time() - start) * 1000
            run_record["total_ms"] = round(elapsed_ms, 2)
            self.metrics.record("latency_ms", elapsed_ms)
            self.metrics.record("steps_count", len(run_record["steps"]))
            root_span.set_attribute("status", run_record["status"])
            root_span.set_attribute("total_ms", run_record["total_ms"])
            
            self.logger.info(
                "Run completed",
                trace_id=trace_id,
                status=run_record["status"],
                total_ms=run_record["total_ms"],
            )
        
        self.runs.append(run_record)
        return run_record
    
    def _execute_step(self, trace_id: str, step_name: str, func, *args) -> dict:
        """Execute a single step with tracing, logging, and metrics."""
        with self.tracer.start_as_current_span(step_name) as span:
            t0 = time.time()
            self.logger.debug(f"Step '{step_name}' started", trace_id=trace_id, step=step_name)
            
            result = func(*args)
            
            duration_ms = (time.time() - t0) * 1000
            span.set_attribute("duration_ms", round(duration_ms, 2))
            self.metrics.record(f"step.{step_name}.latency_ms", duration_ms)
            self.logger.debug(
                f"Step '{step_name}' completed",
                trace_id=trace_id, step=step_name, duration_ms=round(duration_ms, 2),
            )
            
            return {"step": step_name, "duration_ms": round(duration_ms, 2), "result": result}
    
    # ── Simulated agent steps ────────────────────────────────────────────────
    
    def _plan(self, query: str) -> str:
        time.sleep(random.uniform(0.02, 0.08))
        return f"Plan: retrieve docs about '{query}', then synthesize answer."
    
    def _retrieve(self, query: str) -> list[str]:
        time.sleep(random.uniform(0.03, 0.1))
        return [
            "Q3 revenue: $4.2B, up 12% YoY.",
            "Operating margin improved to 18.3%.",
            "Guidance raised for Q4.",
        ]
    
    def _reason(self, query: str, docs: list[str]) -> str:
        time.sleep(random.uniform(0.05, 0.2))
        tokens = random.randint(200, 450)
        self.metrics.record("tokens", tokens)
        # Simulate occasional failures
        if random.random() < 0.15:
            raise RuntimeError("LLM timeout after 30s")
        return f"Q3 was strong: revenue up 12% to $4.2B with 18.3% margins. Outlook positive."


print("ObservableAgent class defined.")

In [ ]:
# ── Run the observable agent 20 times ────────────────────────────────────────

exporter.clear()  # reset OTel spans
random.seed(123)

agent = ObservableAgent("earnings-analyst")

queries = [
    "Summarize Q3 earnings",
    "What was the revenue growth?",
    "Compare margins to last quarter",
    "What is the Q4 outlook?",
    "Identify key risk factors",
] * 4  # 20 runs

for q in queries:
    result = agent.run(q)
    status = result["status"]
    ms = result["total_ms"]
    print(f"  [{status:7s}] {ms:6.1f}ms  {q}")

print(f"\nCompleted {len(queries)} runs.")

In [ ]:
# ── Review collected metrics ─────────────────────────────────────────────────

print("=== Agent Metrics Summary ===")
print(json.dumps(agent.metrics.summary(), indent=2))

---
## 6. The Observability Stack

In production, you combine multiple tools. Here is the standard stack:

```
+------------------+     +-------------------+     +-----------------+
|  Your Agent      |     |  Collector /      |     |  Visualization  |
|                  |---->|  Backend          |---->|                 |
|  OTel SDK        |     |  Jaeger / Tempo   |     |  Grafana        |
|  Structured Logs |     |  Loki / ELK       |     |  Dashboards     |
|  Metrics Client  |     |  Prometheus       |     |  Alerts         |
+------------------+     +-------------------+     +-----------------+
```

| Layer | Tool | What it does |
|-------|------|--------------|
| **Tracing** | OpenTelemetry + Jaeger/Tempo | Distributed trace collection and visualization |
| **Logging** | Structured JSON + Loki/ELK | Searchable, aggregatable log storage |
| **Metrics** | Prometheus + Grafana | Time-series metrics, dashboards, alerts |
| **AI-specific** | LangSmith / Langfuse | LLM-aware tracing with prompt/completion capture |

Let's simulate Prometheus-style metric exposition.

In [ ]:
# ── Prometheus-style metric exposition ───────────────────────────────────────
# In production, a /metrics endpoint would serve this text.

def to_prometheus_format(collector: MetricsCollector) -> str:
    """Convert metrics to Prometheus exposition format."""
    lines = []
    
    for name, value in collector.counters.items():
        prom_name = name.replace(".", "_")
        lines.append(f"# TYPE {prom_name} counter")
        lines.append(f"{prom_name} {value}")
    
    for name, values in collector.histograms.items():
        if not values:
            continue
        prom_name = name.replace(".", "_")
        lines.append(f"# TYPE {prom_name} summary")
        lines.append(f'{prom_name}{{quantile="0.5"}} {collector.get_percentile(name, 50):.2f}')
        lines.append(f'{prom_name}{{quantile="0.95"}} {collector.get_percentile(name, 95):.2f}')
        lines.append(f'{prom_name}{{quantile="0.99"}} {collector.get_percentile(name, 99):.2f}')
        lines.append(f"{prom_name}_count {len(values)}")
        lines.append(f"{prom_name}_sum {sum(values):.2f}")
    
    for name, value in collector.gauges.items():
        prom_name = name.replace(".", "_")
        lines.append(f"# TYPE {prom_name} gauge")
        lines.append(f"{prom_name} {value}")
    
    return "\n".join(lines)


print("=== Prometheus Exposition Format ===")
print(to_prometheus_format(agent.metrics))

---
## 7. Debugging Agent Behavior

When an agent fails, the trace is your best friend. Let's build a trace analyzer that surfaces the root cause.

In [ ]:
# ── Trace Analyzer ───────────────────────────────────────────────────────────

class TraceAnalyzer:
    """Analyze agent run records to find failures and bottlenecks."""
    
    def __init__(self, runs: list[dict]):
        self.runs = runs
    
    def failed_runs(self) -> list[dict]:
        """Return all failed runs with error details."""
        return [r for r in self.runs if r["status"] == "error"]
    
    def slowest_runs(self, n: int = 5) -> list[dict]:
        """Return the N slowest runs."""
        return sorted(self.runs, key=lambda r: r.get("total_ms", 0), reverse=True)[:n]
    
    def bottleneck_analysis(self) -> dict:
        """Find which step contributes most to total latency."""
        step_times = defaultdict(list)
        for run in self.runs:
            for step in run.get("steps", []):
                step_times[step["step"]].append(step["duration_ms"])
        
        result = {}
        for step_name, times in step_times.items():
            result[step_name] = {
                "mean_ms": round(statistics.mean(times), 2),
                "max_ms": round(max(times), 2),
                "p95_ms": round(sorted(times)[int(len(times) * 0.95)], 2) if len(times) > 1 else round(times[0], 2),
                "count": len(times),
            }
        return result
    
    def error_report(self) -> str:
        """Generate a human-readable error report."""
        failed = self.failed_runs()
        if not failed:
            return "No failures detected."
        
        lines = [f"=== Error Report: {len(failed)} failures out of {len(self.runs)} runs ===", ""]
        for run in failed:
            lines.append(f"  Trace: {run['trace_id']}")
            lines.append(f"  Query: {run['query']}")
            lines.append(f"  Error: {run.get('error', 'unknown')}")
            lines.append(f"  Steps completed: {len(run.get('steps', []))}")
            if run.get("steps"):
                last_step = run["steps"][-1]
                lines.append(f"  Last successful step: {last_step['step']} ({last_step['duration_ms']}ms)")
            lines.append("")
        return "\n".join(lines)


analyzer = TraceAnalyzer(agent.runs)

# Error report
print(analyzer.error_report())

In [ ]:
# ── Bottleneck analysis ──────────────────────────────────────────────────────

print("=== Bottleneck Analysis ===")
bottlenecks = analyzer.bottleneck_analysis()
print(json.dumps(bottlenecks, indent=2))

print("\n=== Interpretation ===")
if bottlenecks:
    slowest = max(bottlenecks.items(), key=lambda x: x[1]["mean_ms"])
    print(f"The slowest step is '{slowest[0]}' with mean latency of {slowest[1]['mean_ms']}ms.")
    print("This is your primary optimization target.")

In [ ]:
# ── Slowest runs ─────────────────────────────────────────────────────────────

print("=== Top 5 Slowest Runs ===")
for run in analyzer.slowest_runs(5):
    steps_str = " -> ".join(
        f"{s['step']}({s['duration_ms']}ms)" for s in run.get("steps", [])
    )
    print(f"  [{run['status']:7s}] {run['total_ms']:6.1f}ms  {run['trace_id']}  {steps_str}")

---
## 8. Building Dashboards

In production you would use Grafana. Here we build a text-based dashboard that summarizes agent health at a glance.

In [ ]:
# ── Text-based Agent Dashboard ───────────────────────────────────────────────

def agent_dashboard(agent: ObservableAgent) -> str:
    """Generate a text-based dashboard from agent observability data."""
    m = agent.metrics
    s = m.summary()
    total = s["counters"].get("runs.total", 0)
    success = s["counters"].get("runs.success", 0)
    errors = s["counters"].get("runs.error", 0)
    success_rate = (success / total * 100) if total else 0
    
    lat = s["histograms"].get("latency_ms", {})
    tok = s["histograms"].get("tokens", {})
    
    lines = [
        "=" * 60,
        f"  AGENT DASHBOARD: {agent.name}",
        f"  Generated: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}",
        "=" * 60,
        "",
        "  HEALTH",
        "  " + "-" * 40,
        f"  Total runs:    {total}",
        f"  Successes:     {success}",
        f"  Errors:        {errors}",
        f"  Success rate:  {success_rate:.1f}%  {'[OK]' if success_rate >= 95 else '[WARNING]' if success_rate >= 80 else '[CRITICAL]'}",
        "",
        "  LATENCY (ms)",
        "  " + "-" * 40,
    ]
    
    if lat:
        lines += [
            f"  p50:  {lat['p50']:>8.1f}",
            f"  p95:  {lat['p95']:>8.1f}",
            f"  p99:  {lat['p99']:>8.1f}",
            f"  max:  {lat['max']:>8.1f}",
        ]
    
    lines += ["", "  TOKEN USAGE", "  " + "-" * 40]
    if tok:
        lines += [
            f"  Mean:  {tok['mean']:>8.1f}",
            f"  p95:   {tok['p95']:>8.1f}",
            f"  Total: {tok['count'] * tok['mean']:>8.0f}  (estimated)",
        ]
    
    # Latency histogram (ASCII bar chart)
    if m.histograms.get("latency_ms"):
        values = m.histograms["latency_ms"]
        buckets = [100, 200, 300, 400, 500, 600]
        lines += ["", "  LATENCY DISTRIBUTION", "  " + "-" * 40]
        prev = 0
        for b in buckets:
            count = sum(1 for v in values if prev <= v < b)
            bar = "#" * count
            lines.append(f"  {prev:>4}-{b:<4}ms | {bar} ({count})")
            prev = b
        overflow = sum(1 for v in values if v >= buckets[-1])
        if overflow:
            lines.append(f"  {buckets[-1]:>4}+   ms | {'#' * overflow} ({overflow})")
    
    # Recent errors
    error_logs = agent.logger.get_logs(level="ERROR")
    if error_logs:
        lines += ["", "  RECENT ERRORS", "  " + "-" * 40]
        for entry in error_logs[-5:]:
            lines.append(f"  [{entry['trace_id']}] {entry['message']}")
    
    lines.append("\n" + "=" * 60)
    return "\n".join(lines)


print(agent_dashboard(agent))

In [ ]:
# ── Step-level dashboard ─────────────────────────────────────────────────────

def step_dashboard(agent: ObservableAgent) -> str:
    """Dashboard focused on per-step performance."""
    analyzer = TraceAnalyzer(agent.runs)
    bottlenecks = analyzer.bottleneck_analysis()
    
    lines = [
        "=" * 60,
        "  STEP PERFORMANCE BREAKDOWN",
        "=" * 60,
        "",
        f"  {'Step':<15} {'Mean(ms)':>10} {'P95(ms)':>10} {'Max(ms)':>10} {'Count':>8}",
        "  " + "-" * 55,
    ]
    
    total_mean = 0
    for step_name, stats in sorted(bottlenecks.items()):
        lines.append(
            f"  {step_name:<15} {stats['mean_ms']:>10.1f} {stats['p95_ms']:>10.1f} "
            f"{stats['max_ms']:>10.1f} {stats['count']:>8}"
        )
        total_mean += stats["mean_ms"]
    
    lines.append("  " + "-" * 55)
    lines.append(f"  {'TOTAL':<15} {total_mean:>10.1f}")
    
    # Percentage breakdown
    if total_mean > 0:
        lines += ["", "  TIME SHARE"]
        for step_name, stats in sorted(bottlenecks.items(), key=lambda x: x[1]["mean_ms"], reverse=True):
            pct = stats["mean_ms"] / total_mean * 100
            bar = "#" * int(pct / 2)
            lines.append(f"  {step_name:<15} {bar} {pct:.0f}%")
    
    lines.append("\n" + "=" * 60)
    return "\n".join(lines)


print(step_dashboard(agent))

---
## 9. LangSmith and Langfuse Integration Patterns

LangSmith and Langfuse are purpose-built for LLM observability. They capture prompts, completions, token counts, and latency with minimal code changes.

Below we show the **integration patterns** without requiring API keys. These are the decorators and wrappers you would use in a real project.

In [ ]:
# ── LangSmith-style Decorator Pattern ────────────────────────────────────────
# In production, you would: pip install langsmith, set LANGSMITH_API_KEY,
# and use @traceable from langsmith. Here we build a local equivalent.

# This captures the same data LangSmith would, stored locally.

_langsmith_traces: list[dict] = []


def traceable(run_type: str = "chain"):
    """Local equivalent of langsmith.traceable decorator.
    
    In production, replace with:
        from langsmith import traceable
    
    Captures: function name, inputs, outputs, latency, run_type.
    """
    def decorator(func):
        def wrapper(*args, **kwargs):
            run_id = str(uuid.uuid4())[:8]
            start = time.time()
            
            trace_entry = {
                "run_id": run_id,
                "name": func.__name__,
                "run_type": run_type,
                "inputs": {"args": [str(a)[:100] for a in args], "kwargs": {k: str(v)[:100] for k, v in kwargs.items()}},
                "start_time": datetime.now(timezone.utc).isoformat(),
                "status": "success",
            }
            
            try:
                result = func(*args, **kwargs)
                trace_entry["output"] = str(result)[:200]
                return result
            except Exception as e:
                trace_entry["status"] = "error"
                trace_entry["error"] = str(e)
                raise
            finally:
                trace_entry["duration_ms"] = round((time.time() - start) * 1000, 2)
                _langsmith_traces.append(trace_entry)
        
        wrapper.__name__ = func.__name__
        return wrapper
    return decorator


# ── Use the decorator ─────────────────────────────────────────────────────────

@traceable(run_type="retriever")
def retrieve_documents(query: str) -> list[str]:
    time.sleep(0.05)
    return ["Doc A: Revenue data", "Doc B: Margin analysis"]


@traceable(run_type="llm")
def call_llm(prompt: str) -> str:
    time.sleep(0.1)
    return "The company showed strong Q3 results."


@traceable(run_type="chain")
def rag_pipeline(query: str) -> str:
    docs = retrieve_documents(query)
    context = " ".join(docs)
    prompt = f"Given context: {context}\n\nAnswer: {query}"
    return call_llm(prompt)


# Run the pipeline
answer = rag_pipeline("Summarize earnings")
print(f"Answer: {answer}\n")

print("=== Captured Traces (LangSmith-style) ===")
for t in _langsmith_traces:
    print(json.dumps(t, indent=2))
    print()

In [ ]:
# ── Langfuse-style Observation Pattern ───────────────────────────────────────
# Langfuse uses a generation/span model. Here is the local equivalent.

class LangfuseLocalTrace:
    """Local equivalent of Langfuse's trace/generation pattern.
    
    In production, replace with:
        from langfuse import Langfuse
        langfuse = Langfuse()
        trace = langfuse.trace(name="my-agent")
    """
    
    def __init__(self, name: str):
        self.trace_id = str(uuid.uuid4())[:8]
        self.name = name
        self.observations: list[dict] = []
        self.start_time = time.time()
    
    def generation(self, name: str, model: str, input_text: str, output_text: str,
                   tokens_prompt: int = 0, tokens_completion: int = 0):
        """Record an LLM generation (equivalent to langfuse trace.generation())."""
        self.observations.append({
            "type": "generation",
            "name": name,
            "model": model,
            "input": input_text[:200],
            "output": output_text[:200],
            "usage": {"prompt_tokens": tokens_prompt, "completion_tokens": tokens_completion},
            "timestamp": datetime.now(timezone.utc).isoformat(),
        })
    
    def span(self, name: str, input_data: str = "", output_data: str = ""):
        """Record a span (equivalent to langfuse trace.span())."""
        self.observations.append({
            "type": "span",
            "name": name,
            "input": input_data[:200],
            "output": output_data[:200],
            "timestamp": datetime.now(timezone.utc).isoformat(),
        })
    
    def score(self, name: str, value: float, comment: str = ""):
        """Attach a score to the trace (for evaluation)."""
        self.observations.append({
            "type": "score",
            "name": name,
            "value": value,
            "comment": comment,
        })
    
    def summary(self) -> dict:
        return {
            "trace_id": self.trace_id,
            "name": self.name,
            "duration_ms": round((time.time() - self.start_time) * 1000, 2),
            "observations": self.observations,
        }


# ── Use the Langfuse-style pattern ────────────────────────────────────────────
lf_trace = LangfuseLocalTrace(name="rag-agent")

# Simulate a RAG pipeline with Langfuse-style instrumentation
lf_trace.span(name="retrieve", input_data="earnings query", output_data="3 docs retrieved")

lf_trace.generation(
    name="synthesize-answer",
    model="gpt-4o-mini",
    input_text="Context: revenue up 12%. Question: summarize earnings",
    output_text="Q3 showed strong revenue growth of 12% YoY.",
    tokens_prompt=150,
    tokens_completion=25,
)

lf_trace.score(name="relevance", value=0.92, comment="Answer directly addresses the query.")
lf_trace.score(name="faithfulness", value=0.88, comment="Consistent with retrieved documents.")

print("=== Langfuse-style Trace ===")
print(json.dumps(lf_trace.summary(), indent=2))

In [ ]:
# ── Production Integration Cheatsheet ────────────────────────────────────────

cheatsheet = """
=== PRODUCTION INTEGRATION CHEATSHEET ===

1. LangSmith Setup:
   pip install langsmith
   export LANGCHAIN_TRACING_V2=true
   export LANGCHAIN_API_KEY=<your-key>
   
   from langsmith import traceable
   
   @traceable(run_type="chain")
   def my_agent(query): ...

2. Langfuse Setup:
   pip install langfuse
   export LANGFUSE_PUBLIC_KEY=<key>
   export LANGFUSE_SECRET_KEY=<key>
   
   from langfuse import Langfuse
   langfuse = Langfuse()
   trace = langfuse.trace(name="my-agent")
   trace.generation(name="llm-call", model="gpt-4o", ...)

3. OpenTelemetry to Jaeger:
   pip install opentelemetry-exporter-jaeger
   from opentelemetry.exporter.jaeger.thrift import JaegerExporter
   exporter = JaegerExporter(agent_host_name="localhost", agent_port=6831)

4. Key Principle:
   Choose ONE primary tool (LangSmith OR Langfuse) for LLM-specific
   tracing, and supplement with OTel for infrastructure-level tracing.
"""
print(cheatsheet)

---
## 10. Exercises

### Exercise 1: Conceptual Questions

Answer these in a markdown cell below:

1. What is the difference between a **trace** and a **span** in OpenTelemetry?
2. Why are plain-text `print()` logs insufficient for production agents?
3. An agent has a 99% success rate but p95 latency of 12 seconds. Is this acceptable? What would you investigate?
4. You discover that 80% of your agent's latency comes from the retrieval step. Name three strategies to reduce it.
5. When would you choose LangSmith over a generic OpenTelemetry setup?

*Your answers here...*

### Exercise 2: Instrument a RAG Pipeline

Build a simulated RAG pipeline with these steps:
1. **Query rewriting** — rewrite the user query for better retrieval
2. **Retrieval** — fetch documents (simulated)
3. **Reranking** — score and sort documents
4. **Generation** — produce the answer

Instrument it with:
- OpenTelemetry spans for each step
- Structured logging
- Metrics collection (latency per step, total tokens, relevance scores)

Run it 10 times and display a dashboard.

In [ ]:
# ── Exercise 2: Starter Code ─────────────────────────────────────────────────

class ObservableRAGPipeline:
    """TODO: Implement a fully instrumented RAG pipeline."""
    
    def __init__(self):
        self.logger = AgentLogger("rag-pipeline")
        self.metrics = MetricsCollector()
        self.tracer = trace.get_tracer("rag-pipeline")
        self.runs = []
    
    def run(self, query: str) -> dict:
        """Execute the full RAG pipeline with observability."""
        trace_id = str(uuid.uuid4())[:8]
        # TODO: Implement the four steps with full instrumentation:
        # 1. query_rewrite
        # 2. retrieve
        # 3. rerank
        # 4. generate
        #
        # Each step should:
        # - Be wrapped in a tracer span
        # - Emit structured log entries
        # - Record metrics (latency, tokens, relevance)
        pass


# Run 10 times and display dashboard
# pipeline = ObservableRAGPipeline()
# for i in range(10):
#     pipeline.run("What were Q3 earnings?")
# print(agent_dashboard(...))

### Exercise 3: Design — Observability for a Multi-Agent System

You are building a system with three agents:
- **Planner Agent** — breaks a complex task into subtasks
- **Research Agent** — gathers information from multiple sources
- **Writer Agent** — synthesizes findings into a report

Design the observability layer. In a markdown cell below, describe:

1. **Trace structure** — How would you represent the parent-child relationships between agents? Draw the span tree.
2. **Correlation** — How do you link a Writer Agent failure back to a Research Agent retrieval issue?
3. **Metrics** — List 5 metrics specific to multi-agent coordination (beyond single-agent metrics).
4. **Alerting rules** — Define 3 alerting rules with thresholds.
5. **Dashboard layout** — Sketch (in text) a Grafana dashboard with 4 panels for this system.

*Your design here...*

---

## Key Takeaways

1. **Observability is not optional** — without it, agents are black boxes that fail silently.
2. **Three pillars**: traces (what happened), logs (why it happened), metrics (how often and how fast).
3. **OpenTelemetry** is the standard for distributed tracing — learn it once, use it everywhere.
4. **Structured logging** (JSON) enables querying and aggregation; `print()` does not.
5. **Focus on actionable metrics**: latency percentiles, success rate, token usage, step-level breakdown.
6. **LangSmith/Langfuse** add LLM-specific observability (prompt/completion capture, evaluation scores).
7. **Instrument early** — retrofitting observability is painful. Build it in from day one.

---
*End of Chapter 13 Lab*